In [ ]:
!python -m pip install -U python-woc pandas matplotlib

In [1]:
import io
import pandas as pd

NETID_VALIDATOR = "amart170"

# Raw peer data provided for validation
raw_peer_data = """Project;#commits;#authors;From;To;#stars;#forks;LastGHCommitDate;LongestGapStart;LongestGapEnd;LongestGapLength;NumCommitsAfterLastGap;ActivityPattern;NumberOfGapsInTimeline;BeforeThemes;AfterThemes;HypothesizedGapReason;HasRecovered;WhyRecovered;WhoRecovered;Currentstatus;RecentThemes;Notes
fcambus_telize;338;13;1377084651;1750791632;873;138;2026/07/09;2024-01;2025-05;17;3;declining;9;Other:Dependency updates;Dependency updates:Automated bot contributions;Likely a feature-complete tool in maintenance mode run by one maintainer. After the C-to-Go rewrite and 4.0.0 version prep (Nov 2022) and a HEAD-request tweak (Nov 2023), no commit message points to unfinished work.;Yes;A Dependabot PR bumping go-chi/chi (2025-06-20) was followed days later by the maintainer bumping Go to 1.23 and updating dependencies.;Same maintainer (Frederic Cambus) plus dependabot[bot];Active;Dependency updates:Other;Telize was rewritten in Go, then went quiet for 17 months after a small HEAD-request feature. Activity resumed with a Dependabot bump and a routine dependency refresh by the same maintainer, so the gap looks like maintenance mode rather than abandonment. Only 3 commits follow the gap in the WoC data, and GitHub shows a later commit (2026/07/09).
fkong7_2dunet;397;9;1553211731;1748543205;15;1;2025/02/28;2022-03;2024-04;26;7;declining;3;Feature development:Other;Feature development:Bug fixes;"Single-developer research code. Pre-gap commits include merges from the Auto-LV-Modeling repo and vague messages (""Updates"", ""Path changes""), which suggests work had moved to that other repo.";Yes;May 2024 commits add optional scaling, a natural-sort bug fix, mesh-motion improvements and cubic-spline interpolation, i.e. new simulation features for the same pipeline.;Same developer (Fanwei Kong);Inactive;Feature development:Bug fixes;"The 26-month gap follows a stretch where the lone developer was merging from another repo (Auto-LV-Modeling) with terse messages. The same developer came back in May 2024 with feature and bug-fix work, then slowed again; GitHub shows the last commit on 2025/02/28, so it is marked Inactive. The cause is inferred, not stated in any commit."
geodynamics_aspect;23494;347;1318199520;1776492642;311;277;2026/09/28;;;0;23494;cyclical;0;N/A;N/A;N/A active Project;N/A;N/A;N/A;Active;Other:Bug fixes;ASPECT has no gap of 3 or more months: it has 23,494 commits from 347 authors and roughly 1,600-2,200 commits every year since 2014. Recent history is mostly PR merges and CI runner fixes, which shows an actively maintained community project.
hyperopt_hyperopt-sklearn;660;53;1361290194;1742634354;1647;273;2025/03/22;2014-11;2016-04;18;464;irregular;9;Feature development:Bug fixes;Feature development:Bug fixes;The original research authors (Bergstra, Komer, Warde-Farley) were active in 2014 and went quiet after the regression support commit (Oct 2014). The commit messages do not state a reason.;Yes;In May 2016 Komer merged upstream and updated the SVM gamma parameter for new scikit-learn syntax, then Li Shen added regression and time-series lag selectors and Chandler Watson ran 2to3 for Python 3.;Mixed: original author Brent Komer restarted it, but new contributors (Li Shen, Chandler Watson) did most of the work;Inactive;Dependency updates:Other;After early work by the research authors in 2014, the project sat idle for 18 months. It revived in 2016 when scikit-learn API changes and new contributors (regression and Python 3 support) pulled it back. A 2025 burst by a new maintainer (sklearn 1.5/1.6 support, new PyPI version) ended on 2025/03/22, just before the cutoff, so it is marked Inactive.
maxulysse_nf-core_modules;8188;619;1564131265;1761924543;1;0;2026/09/11;2019-08;2019-11;4;8180;rising;1;Documentation updates:Feature development;Other:Feature development;Looks like a short pause after the repo was scaffolded (first commit, README, FastQC prototype, Slack link fix, all in late July 2019). The 4-month gap is the shortest of the set.;Yes;The Dec 2019 commits add GitHub Actions CI plus the first real module files (FastQC, samtools sort/index, TrimGalore), so the repo moved from scaffold to working modules.;Same core team (Phil Ewels, who also appears before the gap, and Harshil Patel/drpatelh);Active;Feature development:Dependency updates;This is a fork (1 star, 0 forks) of the nf-core modules repo, so its history is really the upstream project. The only gap of 3+ months is a 4-month pause after the July 2019 scaffolding, ended by CI setup and the first modules from the same core team. Activity then grew steadily to 8,188 commits and 619 authors.
mpmccode_mpmc;654;43;1330027139;1761071129;53;13;2025/10/21;2024-12;2025-09;10;4;irregular;10;Bug fixes:Feature development;Bug fixes:Other;"The pre-gap commits are a burst of merging ""Jeff's changes"" and fixing CUDA and LAPACK build errors by one developer, and activity then stops after Nov 2024. The messages give no explicit reason.";Yes (minimal);Two pull requests (#23, #24) on 2025-10-21 corrected input files for c2h4 and c2h4p (fixed carbon masses).;A new contributor (Angela, via PRs) merged by the existing maintainer Adam Hogan;Active;Bug fixes:Other;A small research-code project where one developer pushed build and CUDA fixes in a burst, then went quiet for 10 months. The restart is only 4 commits on a single day, from a new contributor fixing input data, so recovery is weak. GitHub shows 2025/10/21 as the last commit, which keeps it Active under the 2025-09-30 rule.
ohdsi_webapi;7861;211;1417636192;1762475208;152;184;2026/09/28;;;0;7861;declining;0;N/A;N/A;N/A active Project;N/A;N/A;N/A;Active;Automated bot contributions:Dependency updates;WebAPI has no gap of 3 or more months across 7,861 commits and 211 authors. Activity peaked in 2018-2019 (about 1,500-1,700 commits a year), fell to 252 in 2023, rebounded to 622 in 2024, and recent history is dominated by Dependabot updates with occasional features (Snowflake key-pair authentication).
omniscale_imposm3;1270;71;1356527250;1762291878;774;161;2026/08/05;2021-11;2022-06;8;95;declining;6;Other;Other:Dependency updates;"The 10 commits before the gap are all auto-generated ""Merge <sha> into <sha>"" pull-request refs from a single day (2021-10-12) and say nothing about the project. The real cause is not visible in commit messages.";Yes;A Travis-to-GitHub-Actions migration with tests against multiple Postgres/PostGIS versions (Aug 2022), then test updates and protobuf dependency updates.;Mixed: François de Metz (earlier PR contributor) and an outside contributor (dflo) restarted it, and the original maintainer Oliver Tonnhofer returned within weeks;Active;Bug fixes:Other;The commits before the gap are synthetic PR-merge refs, so the commit messages cannot explain the 8-month pause. It recovered in mid-2022 through a CI migration (Travis to GitHub Actions) and test fixes, with maintainer Oliver Tonnhofer back on board, and was active with real fixes (tilelist deadlock, PostGIS cleanup) in Sept 2025. The cause needs a check of GitHub issues and PRs from late 2021.
opensafely_long-covid-risk-factors-and-prediction;587;6;1636586992;1732624252;1;0;2024/11/26;2023-08;2024-10;15;1;declining;1;Feature development:Documentation updates;Feature development;"A single-author analysis repo whose last pre-gap commit is ""add preprint link"" (2023-07-11), which suggests the analysis was finished and published as a preprint.";No (one commit only);N/A. The single follow-up commit (2024-11-26) reorders cohorts to match the manuscript, probably tied to a manuscript revision.;Same author (Yinghui Wei);Inactive;Feature development:Documentation updates;This analysis repo went quiet after the preprint link was added in July 2023, which fits a finished project. One later commit aligns cohort order with the manuscript, probably a revision, and nothing has followed (GitHub last commit 2024/11/26), so it is Inactive. It has only one author in practice (6 listed), so the pattern is easy to read as a study lifecycle.
ziotom78_healpix.jl;742;29;1419267990;1758788511;58;20;2026/06/24;2017-04;2017-12;9;692;irregular;11;Dependency updates:Documentation updates;Feature development:Other;The maintainer wrote in the March 2017 commit that the Julia 0.6 syntax update is not backward compatible with 0.5, so he avoided merging it, and the julia06 branch waited until Jan 2018. This is the clearest gap explanation in the set.;Yes;"""Merge branch 'julia06'"" on 2018-01-17, followed by new features (FITS header keywords, map plotting) and a split of the code into smaller files.";Same maintainer (Maurizio Tomasi);Active;Automated bot contributions:Documentation updates;The gap is explained by the maintainer's own commit message: Julia 0.6 changes were held back because they broke 0.5 compatibility, and were merged in Jan 2018. After the gap the same maintainer added features, and later history is mostly Dependabot, version bumps and docs, so the project is Active."""

df_author = pd.read_csv(io.StringIO(raw_peer_data), sep=";")
df_author.columns = [c.strip() for c in df_author.columns]

try:
    df_net = pd.read_csv("net2prj.csv")
    df_net.columns = [c.strip() for c in df_net.columns]
    net_col = [c for c in df_net.columns if "netid" in c.lower()][0]
    woc_col = [c for c in df_net.columns if "woc" in c.lower()][0]
    author_netid = df_net[
        df_net[woc_col] == "fcambus_telize"
    ][net_col].iloc[0]
except Exception:
    author_netid = "peer_owner"

validation_data = {
    "fcambus_telize": {
        "HasRecovered": ("Yes", "Yes", "Dependabot bump on 2025-06-20 followed by Go 1.23 migration and 3 subsequent commits in WoC/GitHub."),
        "WhyRecovered": (
            "A Dependabot PR bumping go-chi/chi (2025-06-20) was followed days later by the maintainer bumping Go to 1.23 and updating dependencies.",
            "Yes",
            "Commit history shows Go toolchain version bump to 1.23 and dependency maintenance via PRs."
        ),
        "ActivityPattern": ("declining", "Yes", "High initial rewrite volume tapering to long dormancy with minimal modern commits."),
        "Currentstatus": ("Active", "Yes", "GitHub reports commit activity through 2026/07/09"),
        "RecentThemes": ("Dependency updates:Other", "Yes", "Top recent commits are dominated by go-chi/chi for dependency bumps and toolchain revisions"),
    },
    "fkong7_2dunet": {
        "HasRecovered": ("Yes", "Yes", "7 commits logged after 26-month gap across May 2024"),
        "WhyRecovered": (
            "May 2024 commits add optional scaling, a natural-sort bug fix basically stuff for new simulation features for the same pipeline.",
            "Yes",
            "Commit messages explicitly document feature additions"
        ),
        "ActivityPattern": ("declining", "Yes", "Early burst of development in 2019-2021 followed by extended breaks and single-month feature burst."),
        "Currentstatus": ("Inactive", "Yes", "Last commit was in February 2025 and there is no subsequent commits within a 1-year window"),
        "RecentThemes": ("Feature development:Bug fixes", "Yes", "Recent post-gap work consists of mesh algorithm features and natural sorting bug fixes"),
    },
    "geodynamics_aspect": {
        "HasRecovered": ("N/A", "Yes", "Project has 0 gaps of across 3 months across 23,494 commits and continuous development since 2014."),
        "WhyRecovered": ("N/A", "Yes", "Never entered a gap"),
        "ActivityPattern": ("steady", "No", "Peer marked cyclical but commit counts seem to me to be sustained each year"),
        "Currentstatus": ("Active", "Yes", "Consistently updated through September 2026 with activity daily"),
        "RecentThemes": ("Other:Bug fixes", "Yes", "Recent commits by GitHub Actions CI runner adjustments and PR merges."),
    },
    "hyperopt_hyperopt-sklearn": {
        "HasRecovered": ("Yes", "Yes", "464 commits followed a 2014-2016 gap seemingly reviving the package with Python 3 support"),
        "WhyRecovered": (
            "Komer seemed to have merged upstream and updated, then Li Shen and Chandler Watson updated things for Python 3",
            "Yes",
            "Commit logs confirm Python 3 2to3 conversion"
        ),
        "ActivityPattern": ("cyclical", "No", "Peer classified irregular; however, I think with multi-year dormancy followed by discrete maintenance bursts fits a more cyclical release pattern"),
        "Currentstatus": ("Inactive", "Yes", "Last observed commit occurred in March 2025 with no subsequent activity in past 18 months."),
        "RecentThemes": ("Dependency updates:Other", "Yes", "2025 maintenance burst focused strictly on compatibility."),
    },
    "maxulysse_nf-core_modules": {
        "HasRecovered": ("Yes", "Yes", "Over 8k commits is following a 4-month  gap in late 2019"),
        "WhyRecovered": (
            "The Dec 2019 commits add GitHub Actions CI plus the first real module files so it seems the repo moved to working modules.",
            "Yes",
            "Git log reveals CI introduction via GitHub Actions and initial bioinformatics module integration."),
        "ActivityPattern": ("rising", "Yes", "Commit volume grew following that initial beginning to thos 8k+ commits."),
        "Currentstatus": ("Active", "Yes", "Active daily contributions through September 2026 across hundreds of other listed contributors"),
        "RecentThemes": ("Feature development:Dependency updates", "Yes", "Recent commits focus on adding community DSL2 modules and biocontainer updates"),
    },
    "mpmccode_mpmc": {
        "HasRecovered": ("No", "No", "Peer marked Yes but I think since they were 4 minor input-file fixes on a single day following 10 months silence it was not a recovery"),
        "WhyRecovered": (
            "Two pull requests corrected input files for c2h4 and c2h4p",
            "Yes",
            "Commit messages cite PR #23 and #24"),
        "ActivityPattern": ("declining", "No", "Peer marked irregular but the timeline is more showing a steep drop from regular CUDA commits to isolated single-day"),
        "Currentstatus": ("Active", "Yes", "Last commit logged in October 2025"),
        "RecentThemes": ("Bug fixes:Other", "Yes", "Recent changes are input correction bug fixes"),
    },
    "ohdsi_webapi": {
        "HasRecovered": ("N/A", "Yes", "Continuous project with 7k commits and zero gaps of months."),
        "WhyRecovered": ("N/A", "Yes", "Project never experienced a gap"),
        "ActivityPattern": ("declining", "Yes", "Commit volume has continually decreased each year"),
        "Currentstatus": ("Active", "Yes", "Maintained through September 2026 with updates"),
        "RecentThemes": ("Automated bot contributions:Dependency updates", "Yes", "Commit history is dominated by automated Java dependencies."),
    },
    "omniscale_imposm3": {
        "HasRecovered": ("Yes", "Yes", "95 commits following a the 8-month gap including migration and bug fixes."),
        "WhyRecovered": (
            "A migration with tests against multiple Postgres/PostGIS versions",
            "Yes",
            "Commit messages explicitly show '.github/workflows' addition replacing Travis CI and PostGIS test matrix updates."),
        "ActivityPattern": ("declining", "Yes", "High initial development in 2013-2017 followed by extended gaps and reactive updates to other things"),
        "Currentstatus": ("Active", "Yes", "Commit activity goes through August 2026 with fixes."),
        "RecentThemes": ("Bug fixes:Other", "Yes", "Recent patches resolve deadlock conditions and memory cleanups."),
    },
    "opensafely_long-covid-risk-factors-and-prediction": {
        "HasRecovered": ("No", "Yes", "Only a single alignment commit occurred after a near 15-month gap so I think project remains not active"),
        "WhyRecovered": (
            "N/A. The single follow-up commit is probably just tied to a revision",
            "Yes",
            "Single follow-up commit message states 'reorder cohorts to match manuscript' but also seemingly with very little change that I can see"),
        "ActivityPattern": ("declining", "Yes", "Active analysis sprint for that publication and then an immediate drop to 0."),
        "Currentstatus": ("Inactive", "Yes", "No commits since November 2024 so the research is probably done"),
        "RecentThemes": ("Feature development:Documentation updates", "Yes", "Last actually active commits added links and tables."),
    },
    "ziotom78_healpix.jl": {
        "HasRecovered": ("Yes", "Yes", "A good 700 commits occurred after that 9-month gap, seemingly to completely restructure the codebase"),
        "WhyRecovered": (
            "'Merge branches as well as new features (FITS header keywords, map plotting) and a split of the code into smaller files",
            "Yes",
            "Commit on branch merging ended the gap once Julia was stabilized."),
        "ActivityPattern": ("cyclical", "No", "Peer marked irregular; however, based on the long dormancy periods and then major Julia version migrations I think is more representative of cyclical upgrades"),
        "Currentstatus": ("Active", "Yes", "Active maintenance as well as bot dependencies through June 2026"),
        "RecentThemes": ("Automated bot contributions:Documentation updates", "Yes", "Recent commits consist of Dependabot updates and Julia documentation"),
    },
}

fields = [
    "HasRecovered",
    "WhyRecovered",
    "ActivityPattern",
    "Currentstatus",
    "RecentThemes",
]

log_rows = []

# Group rows so that for each field, all 10 projects are listed consecutively
for fld in fields:
    for _, row in df_author.iterrows():
        prj = row["Project"]
        author_val = str(row.get(fld, "")).strip()

        val_tuple = validation_data.get(prj, {}).get(fld, ("", "Yes", ""))
        validator_val = val_tuple[0]
        agreement = val_tuple[1]
        evidence = val_tuple[2]

        log_rows.append(
            {
                "WOCProjectID": prj,
                "ProjectOwnerNetID": author_netid,
                "ValidatorNetID": NETID_VALIDATOR,
                "Field": fld,
                "AuthorValue": author_val,
                "ValidatorValue": validator_val,
                "PreDiscussAgreementstatus(Yes/No)": agreement,
                "AdjudicatedValue": "",
                "KeyEvidence": evidence,
                "Comments": "",
            }
        )

# Construct final DataFrame and export to CSV
df_log = pd.DataFrame(log_rows)

out_cols = [
    "WOCProjectID",
    "ProjectOwnerNetID",
    "ValidatorNetID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus(Yes/No)",
    "AdjudicatedValue",
    "KeyEvidence",
    "Comments",
]

output_filename = f"{NETID_VALIDATOR}_validation_log.csv"
df_log[out_cols].to_csv(output_filename, sep=";", index=False)
print(f"Generated {output_filename} with {len(df_log)} rows grouped by field.")

# Display sample of the first two fields
display(df_log[out_cols].head(20))

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (1292642296.py, line 17)

In [3]:
import io
import pandas as pd

NETID_VALIDATOR = "amart170"
PROJECT_OWNER_NETID = "ttb615"
OUTPUT_FILE = f"{NETID_VALIDATOR}_validation_log.csv"

# Raw peer data provided for validation (quotes escaped)
raw_peer_data = """Project;#commits;#authors;From;To;#stars;#forks;LastGHCommitDate;LongestGapStart;LongestGapEnd;LongestGapLength;NumCommitsAfterLastGap;ActivityPattern;NumberOfGapsInTimeline;BeforeThemes;AfterThemes;HypothesizedGapReason;HasRecovered;WhyRecovered;WhoRecovered;Currentstatus;RecentThemes;Notes
fcambus_telize;338;13;1377084651;1750791632;873;138;2026/07/09;2024-01;2025-05;17;3;declining;9;Other:Dependency updates;Dependency updates:Automated bot contributions;Likely a feature-complete tool in maintenance mode run by one maintainer. After the C-to-Go rewrite and 4.0.0 version prep (Nov 2022) and a HEAD-request tweak (Nov 2023), no commit message points to unfinished work.;Yes;A Dependabot PR bumping go-chi/chi (2025-06-20) was followed days later by the maintainer bumping Go to 1.23 and updating dependencies.;Same maintainer (Frederic Cambus) plus dependabot[bot];Active;Dependency updates:Other;Telize was rewritten in Go, then went quiet for 17 months after a small HEAD-request feature. Activity resumed with a Dependabot bump and a routine dependency refresh by the same maintainer, so the gap looks like maintenance mode rather than abandonment. Only 3 commits follow the gap in the WoC data, and GitHub shows a later commit (2026/07/09).
fkong7_2dunet;397;9;1553211731;1748543205;15;1;2025/02/28;2022-03;2024-04;26;7;declining;3;Feature development:Other;Feature development:Bug fixes;"Single-developer research code. Pre-gap commits include merges from the Auto-LV-Modeling repo and vague messages (""Updates"", ""Path changes""), which suggests work had moved to that other repo.";Yes;May 2024 commits add optional scaling, a natural-sort bug fix, mesh-motion improvements and cubic-spline interpolation, i.e. new simulation features for the same pipeline.;Same developer (Fanwei Kong);Inactive;Feature development:Bug fixes;"The 26-month gap follows a stretch where the lone developer was merging from another repo (Auto-LV-Modeling) with terse messages. The same developer came back in May 2024 with feature and bug-fix work, then slowed again; GitHub shows the last commit on 2025/02/28, so it is marked Inactive. The cause is inferred, not stated in any commit."
geodynamics_aspect;23494;347;1318199520;1776492642;311;277;2026/09/28;;;0;23494;cyclical;0;N/A;N/A;N/A active Project;N/A;N/A;N/A;Active;Other:Bug fixes;ASPECT has no gap of 3 or more months: it has 23,494 commits from 347 authors and roughly 1,600-2,200 commits every year since 2014. Recent history is mostly PR merges and CI runner fixes, which shows an actively maintained community project.
hyperopt_hyperopt-sklearn;660;53;1361290194;1742634354;1647;273;2025/03/22;2014-11;2016-04;18;464;irregular;9;Feature development:Bug fixes;Feature development:Bug fixes;The original research authors (Bergstra, Komer, Warde-Farley) were active in 2014 and went quiet after the regression support commit (Oct 2014). The commit messages do not state a reason.;Yes;In May 2016 Komer merged upstream and updated the SVM gamma parameter for new scikit-learn syntax, then Li Shen added regression and time-series lag selectors and Chandler Watson ran 2to3 for Python 3.;Mixed: original author Brent Komer restarted it, but new contributors (Li Shen, Chandler Watson) did most of the work;Inactive;Dependency updates:Other;After early work by the research authors in 2014, the project sat idle for 18 months. It revived in 2016 when scikit-learn API changes and new contributors (regression and Python 3 support) pulled it back. A 2025 burst by a new maintainer (sklearn 1.5/1.6 support, new PyPI version) ended on 2025/03/22, just before the cutoff, so it is marked Inactive.
maxulysse_nf-core_modules;8188;619;1564131265;1761924543;1;0;2026/09/11;2019-08;2019-11;4;8180;rising;1;Documentation updates:Feature development;Other:Feature development;Looks like a short pause after the repo was scaffolded (first commit, README, FastQC prototype, Slack link fix, all in late July 2019). The 4-month gap is the shortest of the set.;Yes;The Dec 2019 commits add GitHub Actions CI plus the first real module files (FastQC, samtools sort/index, TrimGalore), so the repo moved from scaffold to working modules.;Same core team (Phil Ewels, who also appears before the gap, and Harshil Patel/drpatelh);Active;Feature development:Dependency updates;This is a fork (1 star, 0 forks) of the nf-core modules repo, so its history is really the upstream project. The only gap of 3+ months is a 4-month pause after the July 2019 scaffolding, ended by CI setup and the first modules from the same core team. Activity then grew steadily to 8,188 commits and 619 authors.
mpmccode_mpmc;654;43;1330027139;1761071129;53;13;2025/10/21;2024-12;2025-09;10;4;irregular;10;Bug fixes:Feature development;Bug fixes:Other;"The pre-gap commits are a burst of merging ""Jeff's changes"" and fixing CUDA and LAPACK build errors by one developer, and activity then stops after Nov 2024. The messages give no explicit reason.";Yes (minimal);Two pull requests (#23, #24) on 2025-10-21 corrected input files for c2h4 and c2h4p (fixed carbon masses).;A new contributor (Angela, via PRs) merged by the existing maintainer Adam Hogan;Active;Bug fixes:Other;A small research-code project where one developer pushed build and CUDA fixes in a burst, then went quiet for 10 months. The restart is only 4 commits on a single day, from a new contributor fixing input data, so recovery is weak. GitHub shows 2025/10/21 as the last commit, which keeps it Active under the 2025-09-30 rule.
ohdsi_webapi;7861;211;1417636192;1762475208;152;184;2026/09/28;;;0;7861;declining;0;N/A;N/A;N/A active Project;N/A;N/A;N/A;Active;Automated bot contributions:Dependency updates;WebAPI has no gap of 3 or more months across 7,861 commits and 211 authors. Activity peaked in 2018-2019 (about 1,500-1,700 commits a year), fell to 252 in 2023, rebounded to 622 in 2024, and recent history is dominated by Dependabot updates with occasional features (Snowflake key-pair authentication).
omniscale_imposm3;1270;71;1356527250;1762291878;774;161;2026/08/05;2021-11;2022-06;8;95;declining;6;Other;Other:Dependency updates;"The 10 commits before the gap are all auto-generated ""Merge <sha> into <sha>"" pull-request refs from a single day (2021-10-12) and say nothing about the project. The real cause is not visible in commit messages.";Yes;A Travis-to-GitHub-Actions migration with tests against multiple Postgres/PostGIS versions (Aug 2022), then test updates and protobuf dependency updates.;Mixed: François de Metz (earlier PR contributor) and an outside contributor (dflo) restarted it, and the original maintainer Oliver Tonnhofer returned within weeks;Active;Bug fixes:Other;The commits before the gap are synthetic PR-merge refs, so the commit messages cannot explain the 8-month pause. It recovered in mid-2022 through a CI migration (Travis to GitHub Actions) and test fixes, with maintainer Oliver Tonnhofer back on board, and was active with real fixes (tilelist deadlock, PostGIS cleanup) in Sept 2025. The cause needs a check of GitHub issues and PRs from late 2021.
opensafely_long-covid-risk-factors-and-prediction;587;6;1636586992;1732624252;1;0;2024/11/26;2023-08;2024-10;15;1;declining;1;Feature development:Documentation updates;Feature development;"A single-author analysis repo whose last pre-gap commit is ""add preprint link"" (2023-07-11), which suggests the analysis was finished and published as a preprint.";No (one commit only);N/A. The single follow-up commit (2024-11-26) reorders cohorts to match the manuscript, probably tied to a manuscript revision.;Same author (Yinghui Wei);Inactive;Feature development:Documentation updates;This analysis repo went quiet after the preprint link was added in July 2023, which fits a finished project. One later commit aligns cohort order with the manuscript, probably a revision, and nothing has followed (GitHub last commit 2024/11/26), so it is Inactive. It has only one author in practice (6 listed), so the pattern is easy to read as a study lifecycle.
ziotom78_healpix.jl;742;29;1419267990;1758788511;58;20;2026/06/24;2017-04;2017-12;9;692;irregular;11;Dependency updates:Documentation updates;Feature development:Other;The maintainer wrote in the March 2017 commit that the Julia 0.6 syntax update is not backward compatible with 0.5, so he avoided merging it, and the julia06 branch waited until Jan 2018. This is the clearest gap explanation in the set.;Yes;\"\"\"Merge branch 'julia06'\"\"\" on 2018-01-17, followed by new features (FITS header keywords, map plotting) and a split of the code into smaller files.;Same maintainer (Maurizio Tomasi);Active;Automated bot contributions:Documentation updates;The gap is explained by the maintainer's own commit message: Julia 0.6 changes were held back because they broke 0.5 compatibility, and were merged in Jan 2018. After the gap the same maintainer added features, and later history is mostly Dependabot, version bumps and docs, so the project is Active."""

df_author = pd.read_csv(io.StringIO(raw_peer_data), sep=";")
df_author.columns = [c.strip() for c in df_author.columns]

# Validation data containing ValidatorValue, PreDiscussAgreementstatus(Yes/No), and KeyEvidence
validation_data = {
    "fcambus_telize": {
        "HasRecovered": ("Yes", "Yes", "Dependabot bump on 2025-06-20 followed by Go 1.23 migration and 3 subsequent commits in WoC/GitHub."),
        "WhyRecovered": (
            "A Dependabot PR bumping go-chi/chi (2025-06-20) was followed days later by the maintainer bumping Go to 1.23 and updating dependencies.",
            "Yes",
            "Commit history shows Go toolchain version bump to 1.23 and dependency maintenance via PRs."
        ),
        "ActivityPattern": ("declining", "Yes", "High initial rewrite volume tapering to long dormancy with minimal modern commits."),
        "Currentstatus": ("Active", "Yes", "GitHub reports commit activity through 2026/07/09"),
        "RecentThemes": ("Dependency updates:Other", "Yes", "Top recent commits are dominated by go-chi/chi for dependency bumps and toolchain revisions"),
    },
    "fkong7_2dunet": {
        "HasRecovered": ("Yes", "Yes", "7 commits logged after 26-month gap across May 2024"),
        "WhyRecovered": (
            "May 2024 commits add optional scaling, a natural-sort bug fix basically stuff for new simulation features for the same pipeline.",
            "Yes",
            "Commit messages explicitly document feature additions"
        ),
        "ActivityPattern": ("declining", "Yes", "Early burst of development in 2019-2021 followed by extended breaks and single-month feature burst."),
        "Currentstatus": ("Inactive", "Yes", "Last commit was in February 2025 and there is no subsequent commits within a 1-year window"),
        "RecentThemes": ("Feature development:Bug fixes", "Yes", "Recent post-gap work consists of mesh algorithm features and natural sorting bug fixes"),
    },
    "geodynamics_aspect": {
        "HasRecovered": ("N/A", "Yes", "Project has 0 gaps of across 3 months across 23,494 commits and continuous development since 2014."),
        "WhyRecovered": ("N/A", "Yes", "Never entered a gap"),
        "ActivityPattern": ("steady", "No", "Peer marked cyclical but commit counts seem to me to be sustained each year"),
        "Currentstatus": ("Active", "Yes", "Consistently updated through September 2026 with activity daily"),
        "RecentThemes": ("Other:Bug fixes", "Yes", "Recent commits by GitHub Actions CI runner adjustments and PR merges."),
    },
    "hyperopt_hyperopt-sklearn": {
        "HasRecovered": ("Yes", "Yes", "464 commits followed a 2014-2016 gap seemingly reviving the package with Python 3 support"),
        "WhyRecovered": (
            "Komer seemed to have merged upstream and updated, then Li Shen and Chandler Watson updated things for Python 3",
            "Yes",
            "Commit logs confirm Python 3 2to3 conversion"
        ),
        "ActivityPattern": ("cyclical", "No", "Peer classified irregular; however, I think with multi-year dormancy followed by discrete maintenance bursts fits a more cyclical release pattern"),
        "Currentstatus": ("Inactive", "Yes", "Last observed commit occurred in March 2025 with no subsequent activity in past 18 months."),
        "RecentThemes": ("Dependency updates:Other", "Yes", "2025 maintenance burst focused strictly on compatibility."),
    },
    "maxulysse_nf-core_modules": {
        "HasRecovered": ("Yes", "Yes", "Over 8k commits is following a 4-month  gap in late 2019"),
        "WhyRecovered": (
            "The Dec 2019 commits add GitHub Actions CI plus the first real module files so it seems the repo moved to working modules.",
            "Yes",
            "Git log reveals CI introduction via GitHub Actions and initial bioinformatics module integration."
        ),
        "ActivityPattern": ("rising", "Yes", "Commit volume grew following that initial beginning to thos 8k+ commits."),
        "Currentstatus": ("Active", "Yes", "Active daily contributions through September 2026 across hundreds of other listed contributors"),
        "RecentThemes": ("Feature development:Dependency updates", "Yes", "Recent commits focus on adding community DSL2 modules and biocontainer updates"),
    },
    "mpmccode_mpmc": {
        "HasRecovered": ("No", "No", "Peer marked Yes but I think since they were 4 minor input-file fixes on a single day following 10 months silence it was not a recovery"),
        "WhyRecovered": (
            "Two pull requests corrected input files for c2h4 and c2h4p",
            "Yes",
            "Commit messages cite PR #23 and #24"
        ),
        "ActivityPattern": ("declining", "No", "Peer marked irregular but the timeline is more showing a steep drop from regular CUDA commits to isolated single-day"),
        "Currentstatus": ("Active", "Yes", "Last commit logged in October 2025"),
        "RecentThemes": ("Bug fixes:Other", "Yes", "Recent changes are input correction bug fixes"),
    },
    "ohdsi_webapi": {
        "HasRecovered": ("N/A", "Yes", "Continuous project with 7k commits and zero gaps of months."),
        "WhyRecovered": ("N/A", "Yes", "Project never experienced a gap"),
        "ActivityPattern": ("declining", "Yes", "Commit volume has continually decreased each year"),
        "Currentstatus": ("Active", "Yes", "Maintained through September 2026 with updates"),
        "RecentThemes": ("Automated bot contributions:Dependency updates", "Yes", "Commit history is dominated by automated Java dependencies."),
    },
    "omniscale_imposm3": {
        "HasRecovered": ("Yes", "Yes", "95 commits following a the 8-month gap including migration and bug fixes."),
        "WhyRecovered": (
            "A migration with tests against multiple Postgres/PostGIS versions",
            "Yes",
            "Commit messages explicitly show '.github/workflows' addition replacing Travis CI and PostGIS test matrix updates."
        ),
        "ActivityPattern": ("declining", "Yes", "High initial development in 2013-2017 followed by extended gaps and reactive updates to other things"),
        "Currentstatus": ("Active", "Yes", "Commit activity goes through August 2026 with fixes."),
        "RecentThemes": ("Bug fixes:Other", "Yes", "Recent patches resolve deadlock conditions and memory cleanups."),
    },
    "opensafely_long-covid-risk-factors-and-prediction": {
        "HasRecovered": ("No", "Yes", "Only a single alignment commit occurred after a near 15-month gap so I think project remains not active"),
        "WhyRecovered": (
            "N/A. The single follow-up commit is probably just tied to a revision",
            "Yes",
            "Single follow-up commit message states 'reorder cohorts to match manuscript' but also seemingly with very little change that I can see"
        ),
        "ActivityPattern": ("declining", "Yes", "Active analysis sprint for that publication and then an immediate drop to 0."),
        "Currentstatus": ("Inactive", "Yes", "No commits since November 2024 so the research is probably done"),
        "RecentThemes": ("Feature development:Documentation updates", "Yes", "Last actually active commits added links and tables."),
    },
    "ziotom78_healpix.jl": {
        "HasRecovered": ("Yes", "Yes", "A good 700 commits occurred after that 9-month gap, seemingly to completely restructure the codebase"),
        "WhyRecovered": (
            "'Merge branches as well as new features (FITS header keywords, map plotting) and a split of the code into smaller files",
            "Yes",
            "Commit on branch merging ended the gap once Julia was stabilized."
        ),
        "ActivityPattern": ("cyclical", "No", "Peer marked irregular; however, based on the long dormancy periods and then major Julia version migrations I think is more representative of cyclical upgrades"),
        "Currentstatus": ("Active", "Yes", "Active maintenance as well as bot dependencies through June 2026"),
        "RecentThemes": ("Automated bot contributions:Documentation updates", "Yes", "Recent commits consist of Dependabot updates and Julia documentation"),
    },
}

fields = [
    "HasRecovered",
    "WhyRecovered",
    "ActivityPattern",
    "Currentstatus",
    "RecentThemes",
]

log_rows = []

# Group rows so that for each field, all 10 projects are listed consecutively
for fld in fields:
    for _, row in df_author.iterrows():
        prj = row["Project"]
        author_val = str(row.get(fld, "")).strip()

        val_tuple = validation_data.get(prj, {}).get(fld, ("", "Yes", ""))
        validator_val = val_tuple[0]
        agreement = val_tuple[1]
        evidence = val_tuple[2]

        log_rows.append(
            {
                "WOCProjectID": prj,
                "ProjectOwnerNetID": PROJECT_OWNER_NETID,
                "ValidatorNetID": NETID_VALIDATOR,
                "Field": fld,
                "AuthorValue": author_val,
                "ValidatorValue": validator_val,
                "PreDiscussAgreementstatus(Yes/No)": agreement,
                "AdjudicatedValue": "",
                "KeyEvidence": evidence,
                "Comments": "",
            }
        )

# Construct final DataFrame and export to CSV
df_log = pd.DataFrame(log_rows)

out_cols = [
    "WOCProjectID",
    "ProjectOwnerNetID",
    "ValidatorNetID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "PreDiscussAgreementstatus(Yes/No)",
    "AdjudicatedValue",
    "KeyEvidence",
    "Comments",
]

df_log[out_cols].to_csv(OUTPUT_FILE, sep=";", index=False)
print(f"Generated {OUTPUT_FILE} successfully with {len(df_log)} rows grouped by field.")

# Display preview of results
display(df_log[out_cols].head(15))

Generated amart170_validation_log.csv successfully with 50 rows grouped by field.


,WOCProjectID,ProjectOwnerNetID,ValidatorNetID,Field,AuthorValue,ValidatorValue,PreDiscussAgreementstatus(Yes/No),AdjudicatedValue,KeyEvidence,Comments
0,fcambus_telize,ttb615,amart170,HasRecovered,Yes,Yes,Yes,,Dependabot bump on 2025-06-20 followed by Go 1...,
1,fkong7_2dunet,ttb615,amart170,HasRecovered,Yes,Yes,Yes,,7 commits logged after 26-month gap across May...,
2,geodynamics_aspect,ttb615,amart170,HasRecovered,nan,N/A,Yes,,Project has 0 gaps of across 3 months across 2...,
3,hyperopt_hyperopt-sklearn,ttb615,amart170,HasRecovered,Yes,Yes,Yes,,464 commits followed a 2014-2016 gap seemingly...,
4,maxulysse_nf-core_modules,ttb615,amart170,HasRecovered,Yes,Yes,Yes,,Over 8k commits is following a 4-month gap in...,
5,mpmccode_mpmc,ttb615,amart170,HasRecovered,Yes (minimal),No,No,,Peer marked Yes but I think since they were 4 ...,
6,ohdsi_webapi,ttb615,amart170,HasRecovered,nan,N/A,Yes,,Continuous project with 7k commits and zero ga...,
7,omniscale_imposm3,ttb615,amart170,HasRecovered,Yes,Yes,Yes,,95 commits following a the 8-month gap includi...,
8,opensafely_long-covid-risk-factors-and-prediction,ttb615,amart170,HasRecovered,No (one commit only),No,Yes,,Only a single alignment commit occurred after ...,
9,ziotom78_healpix.jl,ttb615,amart170,HasRecovered,Yes,Yes,Yes,,A good 700 commits occurred after that 9-month...,
